#Proyecto Personal

##Contexto

SuperTienda es una empresa de retail que centraliza informacion sobre clientes, productos, pedidos y operaciones internas. Sin embargo, estos datos se encuentran distribuidos en multiples tablas dentro de una base de datos relacional.

Actualmente, la empresa no cuenta con un dataset consolidado que permita analizar su desempeño de forma clara, lo que dificulta la toma de decisiones estrategicas.

##Objetivo

Explorar la base de datos SuperTienda, identificar relaciones entre tablas y construir un dataset limpio y estructurado que sirva como base para análisis posteriores.

##Enfoque seleccionado

**Enfoque de Rentabilidad**: Combinamos pedidos, detalle de pedidos, productos, clientes y geografia para analizar ingresos, ganancias y márgenes por categoria, región y segmento de cliente.

#Flujo de trabajo

1. Exploración de la base de datos
2. Extraccioón de datos con SQL
3. Limpieza y transformacion con Pandas
4. Exportacion del dataset

#Fase 1: Exploración de la base de datos

Conectamos el notebook a la base de datos y exploramos su estructura para identificar las tablas disponibles, sus columnas y las relaciones entre ellas.

In [1]:
# Descarga de database desde drive automatico

import os
import re
import sqlite3
import requests
import pandas as pd

DB_FILE = 'SuperTienda_Espanol.db'
DRIVE_FILE_ID = '1vzKSdl9raULe-DsycwJTf7rKv1BUU4JA'

def descargar_desde_drive(file_id, output_path):
  url = f'https://drive.google.com/uc?export=download&id={file_id}'
  session = requests.Session()
  response = session.get(url, stream=True)
  match = re.search(r'confirm=([0-9A-Za-z_-]+)', response.text)
  if match:
      response = session.get(f'{url}&confirm={match.group(1)}', stream=True)
  response.raise_for_status()
  with open(output_path, 'wb') as f:
      f.write(response.content)

if not os.path.exists(DB_FILE):
    descargar_desde_drive(DRIVE_FILE_ID, DB_FILE)

In [2]:
# Conexión a la base de datos
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'
%sql sqlite:///SuperTienda_Espanol.db

In [3]:
#Identificamos las tablas disponibles en la base de datos
%%sql
SELECT name
FROM sqlite_master
WHERE type = 'table'

 * sqlite:///SuperTienda_Espanol.db
Done.


name
Clientes
Productos
Geografia
Cliente_Ubicacion
Campanias
Pedidos
Detalle_Pedido
Soporte


La base de datos contiene 8 tablas: `Clientes`, `Productos`, `Geografia`, `Cliente_Ubicacion`, `Campanias`, `Pedidos`, `Detalle_Pedido`, `Soporte`.

A continuacion exploramos las columnas de cada tabla relevante para nuestro enfoque.

In [4]:
%%sql
SELECT *
FROM Pedidos
LIMIT 3;

 * sqlite:///SuperTienda_Espanol.db
Done.


ID_Pedido,ID_Cliente,Fecha_Pedido,Fecha_Envio,Modo_Envio,ID_Campania,Estado_Pedido,Observacion_Interna
ORD-2023-00001,C0174,2025-03-31,2025-04-04,Estándar,None,Completado,Cliente frecuente
ORD-2023-00002,C0254,2024-03-25,2024-03-25,Segunda Clase,MKT005,Completado,Sin observaciones
ORD-2023-00003,C0203,2024-02-20,2024-02-24,Estándar,MKT001,Completado,Revisar descuento aplicado


In [5]:
%%sql
SELECT *
FROM Detalle_Pedido
LIMIT 3;

 * sqlite:///SuperTienda_Espanol.db
Done.


ID_Detalle,ID_Pedido,ID_Producto,Cantidad,Descuento,Ventas,Ganancia,Prioridad
1,ORD-2023-00001,P0082,1,0.0,590.86,322.42,Crítica
2,ORD-2023-00001,P0119,7,0.0,4341.68,1907.85,Media
3,ORD-2023-00001,P0048,2,0.0,1250.86,281.32,Crítica


In [6]:
%%sql
SELECT *
FROM Productos
LIMIT 3;

 * sqlite:///SuperTienda_Espanol.db
Done.


ID_Producto,Nombre_Producto,Categoria,Subcategoria,Precio_Lista,Costo_Unitario,Marca,Activo,Codigo_Barra
P0001,Accesorios Flex 1,Tecnología,Accesorios,445.84,210.5,Alerce,1,7154379937124
P0002,Accesorios Smart 2,Tecnología,Accesorios,10.06,5.92,Vector,1,5756924593752
P0003,Accesorios Plus 3,Tecnología,Accesorios,868.26,685.1,Altura,1,3564651939828


In [7]:
%%sql
SELECT *
FROM Clientes
LIMIT 3;

 * sqlite:///SuperTienda_Espanol.db
Done.


ID_Cliente,Nombre_Cliente,Segmento,Fecha_Registro,Canal_Preferido,Puntaje_Fidelidad,Codigo_Interno
C0001,Fernanda Díaz,Consumidor,2023-02-18,Web,55,CLI-6635-Z
C0002,Valentina Muñoz,Home Office,2023-07-30,Web,58,CLI-2291-Z
C0003,Diego Castillo,Corporativo,2024-03-28,App,100,CLI-2139-X


In [8]:
%%sql
SELECT *
FROM Cliente_Ubicacion
LIMIT 3;

 * sqlite:///SuperTienda_Espanol.db
Done.


ID_Cliente,ID_Ubicacion,Tipo_Direccion,Fecha_Actualizacion
C0001,UB003,Principal,2024-01-08
C0002,UB012,Principal,2024-06-21
C0003,UB013,Principal,2024-05-31


In [9]:
%%sql
SELECT *
FROM Geografia
LIMIT 3;

 * sqlite:///SuperTienda_Espanol.db
Done.


ID_Ubicacion,Pais,Region,Estado,Ciudad,Codigo_Postal,Zona_Logistica,Latitud_Ref
UB001,Chile,Norte,Antofagasta,Antofagasta,1240000,A,-52.1246
UB002,Chile,Norte,Coquimbo,La Serena,1700000,C,-44.4288
UB003,Chile,Norte,Tarapacá,Iquique,1100000,B,-27.2235


##Decisión de tablas

Seleccionamos 6 de las 8 tablas disponibles:

* `Pedidos:`Contiene los pedidos con fecha, modo de envio y estado.
* `Detalle_Pedido:`Contiene el detalle económico de cada pedido: ventas, ganancia y descuento.
* `Productos:`Aporta categoria, subcategoría y nombre del producto.
* `Clientes:`Aporta el segmento comercial del cliente.
* `Clientes_Ubicacion:`Relaciona clientes con su ubicación geográfica.
* `Geografia:`Aporta región y ciudad de cada cliente.


Descartamos `Campanias` y `Soporte` porque no aportan información relevante para el análisis de rentabilidad.

##Fase 2: Extracción de datos con SQL

Construimos una consulta SQL que integra las 6 tablas seleccionadas. El filtro `WHERE cu.Tipo_Direccion = 'Principal'` evita que un cliente con múltiples direcciones registradas aparezca duplicado en el resultado.

In [10]:
%%sql resultado <<
SELECT
p.ID_Pedido,
p.Fecha_Pedido,
p.Modo_Envio,
p.Estado_Pedido,
c.ID_Cliente,
c.Nombre_Cliente,
c.Segmento,
g.Region,
g.Ciudad,
pr.Categoria,
pr.Subcategoria,
pr.Nombre_Producto,
dp.Cantidad,
dp.Descuento,
dp.Ventas,
dp.Ganancia
FROM Pedidos p
JOIN Clientes c ON p.ID_Cliente = c.ID_Cliente
JOIN Cliente_Ubicacion cu ON c.ID_Cliente = cu.ID_Cliente
JOIN Geografia g ON cu.ID_Ubicacion = g.ID_Ubicacion
JOIN Detalle_Pedido dp ON p.ID_Pedido = dp.ID_Pedido
JOIN Productos pr ON dp.ID_Producto = pr.ID_Producto
WHERE cu.Tipo_Direccion = 'Principal'

 * sqlite:///SuperTienda_Espanol.db
Done.
Returning data to local variable resultado


In [11]:
import pandas as pd

# Convertimos el resultado SQL en un DataFrame de Pandas
df = pd.DataFrame(resultado)
df.head()

,ID_Pedido,Fecha_Pedido,Modo_Envio,Estado_Pedido,ID_Cliente,Nombre_Cliente,Segmento,Region,Ciudad,Categoria,Subcategoria,Nombre_Producto,Cantidad,Descuento,Ventas,Ganancia
0,ORD-2023-00001,2025-03-31,Estándar,Completado,C0174,Nicolás Martínez,Corporativo,Sur,Puerto Montt,Oficina,Papelería,Papelería Eco 2,1,0.00,590.86,322.42
1,ORD-2023-00001,2025-03-31,Estándar,Completado,C0174,Nicolás Martínez,Corporativo,Sur,Puerto Montt,Oficina,Sobres,Sobres Plus 9,7,0.00,4341.68,1907.85
2,ORD-2023-00001,2025-03-31,Estándar,Completado,C0174,Nicolás Martínez,Corporativo,Sur,Puerto Montt,Muebles,Sillas,Sillas Plus 8,2,0.00,1250.86,281.32
3,ORD-2023-00001,2025-03-31,Estándar,Completado,C0174,Nicolás Martínez,Corporativo,Sur,Puerto Montt,Muebles,Almacenamiento,Almacenamiento Nova 9,5,0.00,244.35,64.85
4,ORD-2023-00002,2024-03-25,Segunda Clase,Completado,C0254,Isidora Rojas,Consumidor,Sur,Concepción,Oficina,Papelería,Papelería Flex 7,1,0.15,629.31,115.33


#Decisión de columnas

Seleccionamos 16 columnas que permiten analizar rentabilidad desde múltiples ángulos:

* *¿Quién compra?* `Segmento`, `Region`, `Ciudad`
* *¿Qué compra?* `Categoria`, `Subcategoria`, ``Nombre_Producto`
* *¿Cuándo y cómo?* `Fecha_Pedido`, `Modo_Envio`, `Estado_Pedido`
* *¿Cuánto genera?* `Ventas`, `Ganancia`, `Descuento`, `Cantidad`

Descartamos columnas administrativas como `Codigo_Interno`, `Codigo_Barra`, `Observacion_Interna` y `Latitud_Ref` por no aportar valor al análisis de rentabilidad.

##Fase 3: Limpieza y transformación

Con el DataFrame construdo, iniciamos el proceso de preparación. El objetivo es asegurar que los datos sean consistentes, correctamente tipados y enriquecidos con variables calculadas antes de exportar.

###3.1 Inspección técnica inicial

Revisamos la estructura del DataFrame: tipos de dato, cantidad de filas y columnas, y presencia de valores nulos.

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3026 entries, 0 to 3025
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID_Pedido        3026 non-null   object 
 1   Fecha_Pedido     3026 non-null   object 
 2   Modo_Envio       3026 non-null   object 
 3   Estado_Pedido    3026 non-null   object 
 4   ID_Cliente       3026 non-null   object 
 5   Nombre_Cliente   3026 non-null   object 
 6   Segmento         3026 non-null   object 
 7   Region           3026 non-null   object 
 8   Ciudad           3026 non-null   object 
 9   Categoria        3026 non-null   object 
 10  Subcategoria     3026 non-null   object 
 11  Nombre_Producto  3026 non-null   object 
 12  Cantidad         3026 non-null   int64  
 13  Descuento        3026 non-null   float64
 14  Ventas           3026 non-null   float64
 15  Ganancia         3026 non-null   float64
dtypes: float64(3), int64(1), object(12)
memory usage: 378.4+ KB


In [13]:
df.isna().sum()

,0
ID_Pedido,0
Fecha_Pedido,0
Modo_Envio,0
Estado_Pedido,0
ID_Cliente,0
Nombre_Cliente,0
Segmento,0
Region,0
Ciudad,0
Categoria,0


In [14]:
df.describe()

,Cantidad,Descuento,Ventas,Ganancia
count,3026.000000,3026.000000,3026.000000,3026.000000
mean,4.477198,0.099917,1898.627469,595.732905
std,2.278097,0.102992,1556.813776,631.106966
min,1.000000,0.000000,8.550000,-617.810000
25%,3.000000,0.000000,595.525000,125.045000
50%,4.000000,0.100000,1479.955000,381.655000
75%,6.000000,0.200000,2883.220000,884.485000
max,8.000000,0.300000,7133.360000,3651.120000


###3.2 Corrección de tipos de dato

`Fecha_Pedido` llega como texto desde SQL y debe convertirse a formato fecha para poder realizar análisis temporales correctamente.

In [15]:
# Convertimos Fecha_Pedido a tipo datetime
df['Fecha_Pedido'] = pd.to_datetime(df['Fecha_Pedido'])

# Verificamos el cambio
print(df['Fecha_Pedido'].dtype)

datetime64[ns]


###3.3 Estandarización de variables categóricas

Las columnas de texto deben quedar en minúsculas y sin espacios sobrantes. De lo contrario, Python trataría `Norte` y `norte` como dos categorías distintas, rompiendo cualquier agrupación posterior.

In [16]:
# Estandarización: minúsculas + eliminar espacios sobrantes
for col in ['Modo_Envio', 'Estado_Pedido', 'Segmento', 'Region', 'Ciudad', 'Categoria', 'Subcategoria']:
  df[col] = df[col].str.lower().str.strip()

# Verificamos el resultado
print(df['Region'].unique())
print(df['Segmento'].unique())
print(df['Categoria'].unique())

['sur' 'centro' 'austral' 'norte']
['corporativo' 'consumidor' 'home office']
['oficina' 'muebles' 'tecnología']


##3.4 Creación de variables calculadas

Generamos dos nuevas columnas que enriquecen el dataset y facilitan el análisis en el siguiente módulo.

##Variable 1 - Margen de ganancia porcentual:

Expresa la ganancia como porcenaje de las ventas. Permite comparar rentabilidad entre productos y categorías independientemente del volumen de ventas.

In [17]:
# Margen de ganancia: qué porcentaje de las ventas fue ganancia
df['Margen_Pct'] = (df['Ganancia'] / df['Ventas'] * 100).round(2)

print(df['Margen_Pct'].describe())

count    3026.000000
mean       30.008186
std        14.766855
min       -22.010000
25%        20.922500
50%        31.310000
75%        40.890000
max        54.900000
Name: Margen_Pct, dtype: float64


##Variable 2 - Mes del pedido:

Extraemos el mes de la fecha del pedido para fecilitar análisis de tendencias temporales y estabilidad en el siguiente módulo.

In [18]:
# Mes del pedido para análisis temporal
df['Mes_Pedido'] = df['Fecha_Pedido'].dt.month

print(df['Mes_Pedido'].value_counts().sort_index())

Mes_Pedido
1     365
2     287
3     304
4     291
5     233
6     288
7     218
8     218
9     216
10    193
11    204
12    209
Name: count, dtype: int64


##3.5 Verificación final del dataset

Confirmamos que el dataset está limpio, completo y listo para ser exportado.

In [19]:
print(f'Total filas: {len(df)}')
print(f'Total columnas: {len(df.columns)}')
print(f'\nColumnas: {df.columns.tolist()}')
print(f'\nNulos por columna:')
print(df.isna().sum())

Total filas: 3026
Total columnas: 18

Columnas: ['ID_Pedido', 'Fecha_Pedido', 'Modo_Envio', 'Estado_Pedido', 'ID_Cliente', 'Nombre_Cliente', 'Segmento', 'Region', 'Ciudad', 'Categoria', 'Subcategoria', 'Nombre_Producto', 'Cantidad', 'Descuento', 'Ventas', 'Ganancia', 'Margen_Pct', 'Mes_Pedido']

Nulos por columna:
ID_Pedido          0
Fecha_Pedido       0
Modo_Envio         0
Estado_Pedido      0
ID_Cliente         0
Nombre_Cliente     0
Segmento           0
Region             0
Ciudad             0
Categoria          0
Subcategoria       0
Nombre_Producto    0
Cantidad           0
Descuento          0
Ventas             0
Ganancia           0
Margen_Pct         0
Mes_Pedido         0
dtype: int64


In [20]:
df.head(10)

,ID_Pedido,Fecha_Pedido,Modo_Envio,Estado_Pedido,ID_Cliente,Nombre_Cliente,Segmento,Region,Ciudad,Categoria,Subcategoria,Nombre_Producto,Cantidad,Descuento,Ventas,Ganancia,Margen_Pct,Mes_Pedido
0,ORD-2023-00001,2025-03-31,estándar,completado,C0174,Nicolás Martínez,corporativo,sur,puerto montt,oficina,papelería,Papelería Eco 2,1,0.00,590.86,322.42,54.57,3
1,ORD-2023-00001,2025-03-31,estándar,completado,C0174,Nicolás Martínez,corporativo,sur,puerto montt,oficina,sobres,Sobres Plus 9,7,0.00,4341.68,1907.85,43.94,3
2,ORD-2023-00001,2025-03-31,estándar,completado,C0174,Nicolás Martínez,corporativo,sur,puerto montt,muebles,sillas,Sillas Plus 8,2,0.00,1250.86,281.32,22.49,3
3,ORD-2023-00001,2025-03-31,estándar,completado,C0174,Nicolás Martínez,corporativo,sur,puerto montt,muebles,almacenamiento,Almacenamiento Nova 9,5,0.00,244.35,64.85,26.54,3
4,ORD-2023-00002,2024-03-25,segunda clase,completado,C0254,Isidora Rojas,consumidor,sur,concepción,oficina,papelería,Papelería Flex 7,1,0.15,629.31,115.33,18.33,3
5,ORD-2023-00002,2024-03-25,segunda clase,completado,C0254,Isidora Rojas,consumidor,sur,concepción,tecnología,computadores,Computadores Plus 1,2,0.10,1429.92,529.33,37.02,3
6,ORD-2023-00003,2024-02-20,estándar,completado,C0203,Ignacio Araya,corporativo,centro,valparaíso,oficina,archivadores,Archivadores Flex 3,6,0.15,3186.84,1333.80,41.85,2
7,ORD-2023-00003,2024-02-20,estándar,completado,C0203,Ignacio Araya,corporativo,centro,valparaíso,tecnología,computadores,Computadores Eco 9,4,0.30,787.14,-68.77,-8.74,2
8,ORD-2023-00003,2024-02-20,estándar,completado,C0203,Ignacio Araya,corporativo,centro,valparaíso,tecnología,periféricos,Periféricos Flex 4,3,0.20,1007.23,179.29,17.80,2
9,ORD-2023-00004,2023-10-29,primera clase,enviado,C0258,Fernanda Silva,corporativo,centro,talca,oficina,papelería,Papelería Plus 9,8,0.20,2247.17,378.49,16.84,10


##Fase 4: Esportación del dataset

El DataFrame está limpio, estructurado y enriquecido con variables calculadas. Loexportamos en formato CSV para que pueda ser utilizado en herramientas de visualización como Power BI en el siguiente módulo.

In [21]:
df.to_csv('DataSet_Limpio_SuperTienda.csv', index=False)
print('Dataset exportado correstamente.')

Dataset exportado correstamente.


In [22]:
# Solo en Google Colab: descarga el archivo antes de cerrar la sesión
from google.colab import files
files.download('DataSet_Limpio_SuperTienda.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Reflexión analítica

¿Qué tablas decidimos utilizar y por qué?

Utilizamos `Pedidos`, `Detalle_Pedido`, `Productos`, `Clientes`, `Cliente_Ubicacion` y `Geografia`. Estas seis tablas contienen toda la información necesaria para analizar rentabilidad: quién compra, qué compra, cuándo, dónde y cúanto genera.

¿Qué columnas seleccionamos y cuáles descartamos?

Seleccionamos 16 columnas más 2 variables calculadas. Descartamos columnas administrativas como `Codigo_Interno`, `Codigo_Barra`, `Observacion_Interna` y `Latitud_Ref` por ser irrelevantes para el análisis de ventas y ganancias.

¿Qué tipo de análisis permitiriá este dataset en el siguiente módulo?

El dataset permitirá construir dashboards de rentabilidad por categoría de producto, región y segmento de cliente. Con `Margen_Pct` se podrán identificar los productos más y menos rentables. Con `Mes_Pedido` se podrán analizar tendencias temporales de vantas y construir visualizaciones de evolución mensual de ingresos y ganancias.